In [20]:
import numpy as np
import mikeio
import re

def compute_average_slope(dfs1_path):
    """
    Computes the average slope of a cross-shore bathymetry profile from a DFS1 file.
    
    The DFS1 file is assumed to have dimensions ('time', 'x') with a single time step,
    and one data item containing the bathymetry (depth) values.
    
    For a Grid1D geometry, the horizontal distances are computed from grid parameters.
    
    The average slope is defined as:
      (z_last - z_first) / (distance_last - distance_first)
      
    Parameters:
      dfs1_path (str): Path to the DFS1 bathymetry file.
      
    Returns:
      avg_slope (float): Average slope (m/m).
    """
    # Read the DFS1 file.
    ds = mikeio.read(dfs1_path)
    
    # Remove singleton dimensions.
    ds = ds.squeeze()
    
    # If a time dimension still exists, select the first time step.
    if "time" in ds.dims:
        ds = ds.isel(time=0)
    
    # Extract bathymetry values from the first data item.
    item = list(ds)[0]
    bathy = item.to_numpy().ravel()  # Should be shape (n,)
    print("Bathymetry array shape:", bathy.shape)
    
    # Determine horizontal distances using the geometry.
    geom = ds.geometry
    print("Geometry:", geom)
    geom_str = str(geom)
    x = None
    if "Grid1D" in geom_str:
        # Parse n and dx from the string.
        n_match = re.search(r"n\s*=\s*(\d+)", geom_str)
        dx_match = re.search(r"dx\s*=\s*([\d\.]+)", geom_str)
        if n_match and dx_match:
            n = int(n_match.group(1))
            dx = float(dx_match.group(1))
            x = np.arange(n) * dx
            print(f"Parsed Grid1D geometry: n = {n}, dx = {dx}")
        else:
            raise ValueError("Could not parse Grid1D parameters from geometry string.")
    elif "LineString" in geom_str:
        # Fallback: if geometry is a LineString.
        from shapely.geometry import LineString
        if isinstance(geom, LineString):
            coords = list(geom.coords)
            distances = [0.0]
            for i in range(1, len(coords)):
                dx_coord = coords[i][0] - coords[i-1][0]
                dy_coord = coords[i][1] - coords[i-1][1]
                distances.append(distances[-1] + np.sqrt(dx_coord**2 + dy_coord**2))
            x = np.array(distances)
            print("Using LineString geometry for horizontal distances.")
        else:
            raise ValueError("Geometry is not a recognized type.")
    else:
        raise ValueError("Unknown geometry type. Geometry string: " + geom_str)
    
    # Debug prints.
    print("Horizontal distances (m):", x)
    print("Bathymetry values (m):", bathy)
    
    # Compute the average slope over the profile.
    avg_slope = (bathy[-1] - bathy[0]) / (x[-1] - x[0])
    return avg_slope

# Define file paths for the three bathymetry profiles.
north_bathy = r"D:\Phd Research\Wave Model\small scale port lavaca\Stat_dfsu\Cross-shore_bathymetry.dfs1"
west_bathy = r"D:\Phd Research\Wave Model\small scale port lavaca\Stat_dfsu\Cross-shore_bathymetry_west_side.dfs1"
east_bathy = r"D:\Phd Research\Wave Model\small scale port lavaca\Stat_dfsu\Cross-shore_bathymetry_east_side.dfs1"

# Compute average slopes.
north_slope = compute_average_slope(north_bathy)
west_slope = compute_average_slope(west_bathy)
east_slope = compute_average_slope(east_bathy)

print(f"North side average slope: {north_slope:.4f} m/m")
print(f"West side average slope: {west_slope:.4f} m/m")
print(f"East side average slope: {east_slope:.4f} m/m")


Bathymetry array shape: (10,)
Geometry: Grid1D (n=10, dx=219.9)
Parsed Grid1D geometry: n = 10, dx = 219.9
Horizontal distances (m): [   0.   219.9  439.8  659.7  879.6 1099.5 1319.4 1539.3 1759.2 1979.1]
Bathymetry values (m): [ 2.3550708  -0.7023884  -1.0002526  -1.0005976  -1.000816   -1.0007626
 -1.0002202  -0.99889636 -0.9967486  -0.99363184]
Bathymetry array shape: (10,)
Geometry: Grid1D (n=10, dx=202.1)
Parsed Grid1D geometry: n = 10, dx = 202.1
Horizontal distances (m): [   0.   202.1  404.2  606.3  808.4 1010.5 1212.6 1414.7 1616.8 1818.9]
Bathymetry values (m): [-0.10037726 -0.9982005  -1.0000212  -1.0006815  -1.0010313  -1.001198
 -1.001141   -1.0008402  -1.0000602  -0.9988593 ]
Bathymetry array shape: (10,)
Geometry: Grid1D (n=10, dx=152.2)
Parsed Grid1D geometry: n = 10, dx = 152.2
Horizontal distances (m): [   0.   152.2  304.4  456.6  608.8  761.   913.2 1065.4 1217.6 1369.8]
Bathymetry values (m): [ 0.5157229 -0.738598  -1.0002346 -1.0003773 -1.0005295 -1.0006455
 -1.00